# PCB Quality Inspector — Enhanced YOLOv8n

This is the main local Windows/VS Code workflow for the FYP comparison. It verifies the recorded Original YOLOv8n and Enhanced Trial 044 evidence, shows the saved validation results, optionally runs the complete guarded comparison, generates `app.py`, and starts the local six-class PCB inspector.

**Scientific boundary:** the two models use different recorded training authorities, so this is not a controlled same-data ablation. Reported metrics are validation-only. The held-out test split is not used.


In [1]:
# Safe beginner defaults. Change only these values, then click Run All.
RUN_TRAINING = False
RUN_RECOVERY = False
LAUNCH_STREAMLIT = True
RECOVERY_OUTPUT_DIR = None

# Internal verification mode is used only by the repository's notebook check.
import os
PCB_NOTEBOOK_VERIFY_ONLY = os.environ.get("PCB_NOTEBOOK_VERIFY_ONLY") == "1"


## 1. Setup and kernel check

In VS Code, click **Select Kernel** and choose `custom_yolo_pcb\.venv\Scripts\python.exe`. The next cell stops with a friendly message if another interpreter is selected.


In [2]:
import sys
from pathlib import Path
from IPython.display import Markdown, display
import pandas as pd

PACKAGE = Path.cwd().resolve()
if not (PACKAGE / "model_code").is_dir():
    raise RuntimeError("Open PCB_Quality_Inspector.ipynb from the custom_yolo_pcb folder in VS Code.")
sys.path.insert(0, str(PACKAGE / "model_code"))

from generate_app import write_generated_app
from notebook_workflow import (
    DATASET_RELEASE_URL,
    STREAMLIT_URL,
    DatasetUnavailable,
    display_recorded_results,
    ensure_dataset_ready,
    environment_summary,
    run_complete_comparison,
    run_enhanced_recovery,
    start_streamlit,
    stop_streamlit,
    verify_project_kernel,
    verify_publication_inputs,
)

verify_project_kernel(PACKAGE)
print("Project .venv kernel verified.")


Project .venv kernel verified.


In [3]:
environment = environment_summary()
visible_environment = {
    key: value for key, value in environment.items() if key != "python_executable"
}
display(pd.DataFrame(visible_environment.items(), columns=["Environment", "Value"]))


,Environment,Value
0,python,3.11.9
1,pytorch,2.14.0+cpu
2,ultralytics,8.4.84
3,streamlit,1.64.0
4,cuda_available,False
5,device,CPU fallback


## 2. Dataset and publication checks

The notebook reuses a prepared dataset. If it is missing, it verifies and extracts `release-assets/pcb_yolo_train_val_v1.0.0.zip`. If neither is available, it shows the private v1.0.0 Release link and stops before training. No GitHub token is requested or stored.


In [4]:
try:
    dataset_status = ensure_dataset_ready(PACKAGE)
    print(f"Dataset status: {dataset_status['status']}")
except DatasetUnavailable as error:
    dataset_status = None
    print(error)
    if not PCB_NOTEBOOK_VERIFY_ONLY:
        raise


Dataset is not prepared. Download pcb_yolo_train_val_v1.0.0.zip from the private v1.0.0 Release, place it in release-assets, and Run All again: https://github.com/bi19110220-gs/yolov8n-pcb-local-comparison/releases/tag/v1.0.0


In [5]:
publication_hashes = verify_publication_inputs(PACKAGE)
display(pd.DataFrame(publication_hashes.items(), columns=["Verified input", "SHA-256 / status"]))


,Verified input,SHA-256 / status
0,original_best.pt,380243d5f36d6b4e2a37689d02ede81cf045b1d0eec3af...
1,trial044_best.pt,4bdde7e5dc60258de06fceb66ed25fd5814eafa3d6108a...
2,test_split_used,false


## 3. Recorded comparison results

These are the packaged validation results and charts. Opening the notebook does not retrain either model.


In [6]:
recorded = display_recorded_results(PACKAGE)
metric_columns = [
    "model", "training_authority", "precision", "recall", "f1",
    "map50", "map50_95", "short_ap50_95", "inference_latency_ms",
    "parameters", "model_size_mb", "training_seconds",
]
metrics = pd.DataFrame(recorded["metrics"])[metric_columns]
for column in ("precision", "recall", "f1", "map50", "map50_95", "short_ap50_95"):
    metrics[column] = pd.to_numeric(metrics[column]).round(4)
display(metrics)


,model,training_authority,precision,recall,f1,map50,map50_95,short_ap50_95,inference_latency_ms,parameters,model_size_mb,training_seconds
0,original_grouped_v1_yolov8n,grouped-v1,0.9761,0.9732,0.9746,0.9818,0.5551,0.5668,2.637215105377557,3006818,5.968240737915039,14266.4
1,enhanced_trial044_gpu_adaptation,Trial044 OHEM train view plus standard validation,0.9942,0.9919,0.9931,0.9928,0.7585,0.7635,4.771176156978149,3006818,6.037271499633789,7401.07


In [7]:
for chart in recorded["charts"]:
    title = chart.stem.replace("_", " ").title()
    relative_chart = chart.relative_to(PACKAGE).as_posix()
    display(Markdown(f"### {title}"))
    display(Markdown(f"![{title}]({relative_chart})"))


### Original Training

![Original Training](results/charts/original_training.png)

### Trial044 Training

![Trial044 Training](results/charts/trial044_training.png)

### Final Comparison

![Final Comparison](results/charts/final_comparison.png)

### Per Class

![Per Class](results/charts/per_class.png)

## 4. Optional complete local training

Leave `RUN_TRAINING = False` to use the included evidence. Setting it to `True` runs the preserved sequence once: Original training → Original clean validation → Enhanced Trial 044 training → Enhanced clean validation → comparison export. Output goes to a new `results/notebook_runs/YYYYMMDD_HHMMSS/` directory, and detailed logs stay in a sibling local log file instead of filling this notebook.


In [8]:
if RUN_TRAINING:
    if dataset_status is None:
        raise RuntimeError("Prepare the private v1.0.0 dataset before enabling training.")
    completed_output = run_complete_comparison(PACKAGE)
    print(f"Complete comparison saved to: {completed_output.relative_to(PACKAGE)}")
else:
    print("Training skipped (RUN_TRAINING=False). Included weights and results remain unchanged.")


Training skipped (RUN_TRAINING=False). Included weights and results remain unchanged.


In [9]:
if RUN_RECOVERY:
    recovered_output = run_enhanced_recovery(PACKAGE, RECOVERY_OUTPUT_DIR)
    print(f"Enhanced recovery completed in: {recovered_output.relative_to(PACKAGE)}")
else:
    print("Recovery skipped (RUN_RECOVERY=False).")


Recovery skipped (RUN_RECOVERY=False).


## 5. Generate and start the local inspector

The notebook is the canonical source for `app.py`. The generator reproduces the committed file byte-for-byte and refuses to overwrite unrelated handwritten content. The app loads only the included, hash-verified `weights/trial044_best.pt` checkpoint.


In [10]:
generated_app = write_generated_app(PACKAGE / "app.py")
print(f"app.py SHA-256: {generated_app.sha256}")
print("app.py was regenerated." if generated_app.replaced else "app.py already matches the notebook generator.")


app.py SHA-256: dcd89c9842dc7443ac176bdffae12e4de07804a9eb8ed77a6737e7c20424997f
app.py already matches the notebook generator.


In [11]:
if LAUNCH_STREAMLIT and not PCB_NOTEBOOK_VERIFY_ONLY:
    import webbrowser
    streamlit_server = start_streamlit(PACKAGE)
    display(Markdown(f"**Streamlit:** [{STREAMLIT_URL}]({STREAMLIT_URL}) — {streamlit_server['status']}"))
    webbrowser.open(STREAMLIT_URL)
elif PCB_NOTEBOOK_VERIFY_ONLY:
    print("Streamlit launch skipped only for notebook verification; LAUNCH_STREAMLIT remains True.")
else:
    print("Streamlit launch skipped (LAUNCH_STREAMLIT=False).")


Streamlit launch skipped only for notebook verification; LAUNCH_STREAMLIT remains True.


You can also start the identical app later from the VS Code terminal with:

```powershell
& .\.venv\Scripts\python.exe -m streamlit run app.py
```

Open <http://localhost:8501>, upload a JPG/JPEG/PNG image, choose the confidence threshold, and click **Inspect PCB**. The app keeps the upload in memory and provides annotated-PNG and CSV downloads.


In [12]:
# Optional stop control. Change to True and run only this cell when finished.
STOP_STREAMLIT = False
if STOP_STREAMLIT:
    print(stop_streamlit())
else:
    print("Streamlit remains available. Set STOP_STREAMLIT=True and run this cell to stop only the notebook-started process.")


Streamlit remains available. Set STOP_STREAMLIT=True and run this cell to stop only the notebook-started process.


## 6. Interpretation

- **Defects detected — review required** means one or more boxes met the selected confidence threshold.
- **No defects detected** means no boxes met that threshold; it is not a statement that the PCB passed quality control.
- The app uses Enhanced Trial 044 only. The Original model remains in the notebook comparison and training sequence.
- Uploaded-image predictions do not change the recorded validation metrics.
